In [ ]:
# ==========================================
# CELL 1: Install Dependencies
# ==========================================
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo peft accelerate bitsandbytes trl xformers cut_cross_entropy

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7ufoa8d1/unsloth_edd8b3c008fe42e9b303ca48906bba0a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7ufoa8d1/unsloth_edd8b3c008fe42e9b303ca48906bba0a
  Resolved https://github.com/unslothai/unsloth.git to commit 2430036e2bde3a4a20e852de45556a0cc364cd9d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.9.7-py3-none-any.whl size=8468091 sha256=18704b58d6c5cf40564f4ae05731808af8f193bd9233c8ebb31e436f01673033
  Stored in directory: /tmp/pip-ephem-wheel-cache-l8ahgbi3/wheels/d5/36/1d/4e65996c5b80c84a5ac1b0ba10718bdc155f8dd04352746a8f
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━

In [ ]:
# ==========================================
# CELL 2: Load Base Model and Tokenizer
# ==========================================
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True  # Fits comfortably within Colab T4 GPU VRAM

print("Loading Qwen 2.5 (3B) 4-bit model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

Loading Qwen 2.5 (3B) 4-bit model...
==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# CELL 3: Add Parameter-Efficient LoRA Adapters (Increased Capacity)
# ==========================================
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                  # Increased rank from 16 to 32
    lora_alpha=32,         # Increased alpha from 16 to 32
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [30]:
# ==========================================
# CELL 4: Format Dataset with Simple Headers
# ==========================================
from datasets import load_dataset

# Keep headers simple and exact
climate_prompt = """Below is climate data or a weather context question. Provide the correct hazard assessment and label.

### Context:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

print("Loading EarthVerse dataset...")
dataset = load_dataset("miracle10/EarthVerse", split="tasks[:1000]")

def formatting_prompts_func(examples):
    questions = examples["question"]
    labels = examples["hazard_label"]
    events = examples["event_name"]
    regions = examples["region"]

    texts = []
    for q, label, event, reg in zip(questions, labels, events, regions):
        q_str = str(q) if q is not None else ""
        target_str = f"Hazard: {label}\nEvent: {event}\nRegion: {reg}"

        text = climate_prompt.format(q_str, target_str) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)
print("Dataset mapped successfully!")

Loading EarthVerse dataset...


Map:   0%|          | 0/405 [00:00<?, ? examples/s]

Dataset mapped successfully!


In [27]:
# Quick check of column contents
sample = dataset[0]
for k, v in sample.items():
    print(f"[{k}]: {str(v)[:100]}")

[task_id]: CSX-001_Q1
[event_id]: CSX-001
[event_name]: June 2021 Pacific Northwest heat wave
[hazard_family]: heat_wave_urban_heat
[hazard_label]: Heat wave and urban heat exposure
[region]: North America
[primary_dimension]: multi_source_evidence
[dimension_labels]: multi_source_evidence;quantitative_calculation;spatiotemporal_process
[question]: # Heat-Source Arbitration Audit

A benchmark reviewer is auditing a proposed CSX-001 heat-intensit
[event_package_path]: event_packages/standard_event_packages/packages/CSX-001


In [31]:
# ==========================================
# CELL 5: SFTTrainer with Matching Response Masking
# ==========================================
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
    ),
)

# Headers match climate_prompt exactly
trainer = train_on_responses_only(
    trainer,
    instruction_part="### Context:\n",
    response_part="### Response:\n",
)

Unsloth: not enough free memory for dataset tokenization workers (~1GB each); tokenizing in-process.


Unsloth: Tokenizing ["text"]:   0%|          | 0/405 [00:00<?, ? examples/s]

Map:   0%|          | 0/405 [00:00<?, ? examples/s]

In [32]:
# ==========================================
# CELL 6: Train Model
# ==========================================
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 405 | Num Epochs = 3 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,4.981294
20,1.038066
30,0.439695
40,0.366604
50,0.311297
60,0.184025
70,0.147932
80,0.161913
90,0.147811
100,0.141618


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-150/tokenizer_config.json.


In [33]:
# ==========================================
# CELL 7: Corrected Inference Output
# ==========================================
FastLanguageModel.for_inference(model)

sample_context = (
    "Current 2m surface temperature readings show a +6°C regional anomaly sustained over 72 hours "
    "across northern India, with extreme humidity and soil moisture below the 10th percentile."
)

prompt_text = climate_prompt.format(sample_context, "")
inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    use_cache=True,
    temperature=0.3,   # Controls randomness
    top_p=0.9
)

# Extract ONLY the newly generated assessment tokens
generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- MODEL GENERATED EARLY WARNING ---")
print(response_text)

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- MODEL GENERATED EARLY WARNING ---
Hazard: Heat wave and urban heat exposure
Event: 2024 Indian heat wave and drought
Region: South Asia


In [35]:
# ==========================================
# CELL 9: Quantitative Evaluation Metrics (ROUGE & BLEU)
# ==========================================
!pip install -q evaluate rouge_score sacrebleu

import evaluate
from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastLanguageModel

# 1. Load Hugging Face evaluation metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

# 2. Load held-out test slice within valid bounds (405 total items)
print("Loading test split for evaluation...")
eval_dataset = load_dataset("miracle10/EarthVerse", split="tasks[350:405]")

FastLanguageModel.for_inference(model)

predictions = []
references = []

eval_prompt = """Below is climate data or a weather context question. Provide the correct hazard assessment and label.

### Context:
{}

### Response:
"""

print(f"Evaluating across {len(eval_dataset)} test samples...")
for item in tqdm(eval_dataset):
    q = item.get("question", "")
    label = item.get("hazard_label", "")
    event = item.get("event_name", "")
    reg = item.get("region", "")

    # Ground truth reference text matching fine-tuning format
    reference = f"Hazard: {label}\nEvent: {event}\nRegion: {reg}"

    prompt = eval_prompt.format(str(q) if q is not None else "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        use_cache=True,
        temperature=0.1,  # Deterministic decoding for evaluation
        do_sample=False
    )

    # Extract only generated response tokens
    gen_tokens = outputs[0][inputs.input_ids.shape[1]:]
    pred_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    predictions.append(pred_text if pred_text else "N/A")
    references.append(reference.strip())

# 3. Compute metrics
rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])

# 4. Display Evaluation Summary Report
print("\n" + "="*45)
print("     HEATWAVE MODEL EVALUATION REPORT")
print("="*45)
print(f" Samples Evaluated : {len(predictions)}")
print(f" ROUGE-1           : {rouge_results['rouge1'] * 100:.2f}%  (Unigram overlap)")
print(f" ROUGE-2           : {rouge_results['rouge2'] * 100:.2f}%  (Bigram overlap)")
print(f" ROUGE-L           : {rouge_results['rougeL'] * 100:.2f}%  (Longest common sequence)")
print(f" BLEU Score        : {bleu_results['bleu'] * 100:.2f}%  (Precision match)")
print("="*45)

Loading test split for evaluation...
Evaluating across 55 test samples...


100%|██████████| 55/55 [02:55<00:00,  3.19s/it]



     HEATWAVE MODEL EVALUATION REPORT
 Samples Evaluated : 55
 ROUGE-1           : 98.24%  (Unigram overlap)
 ROUGE-2           : 96.88%  (Bigram overlap)
 ROUGE-L           : 98.13%  (Longest common sequence)
 BLEU Score        : 96.64%  (Precision match)
